# AgriSmart AI — Model 2 Field-Domain Adaptation Training
===============================================================
**Goal**: Fine-tune EfficientNet-B2 starting from Model 1 weights (`models/agrismart_best.pth`) using combined PlantVillage + PlantDoc Train data.
**Target Hardware**: Google Colab NVIDIA T4 GPU

### Key Guardrails:
- Model 1 (`models/agrismart_best.pth`) is **UNTOUCHED**.
- PlantDoc TEST (`data/plantdoc/test`) is **LOCKED and UNTOUCHED**.
- PlantDoc oversampling factor: **5.0** via `WeightedRandomSampler`.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))

## 2. Clone / Pull Latest Repository

In [ ]:
import os
if not os.path.exists('/content/agrismart'):
    !git clone https://github.com/Parrthiv125/AgriSmart-AI.git /content/agrismart
%cd /content/agrismart
!git pull origin main

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Obtain PlantVillage Dataset & Reproduce Split

In [ ]:
!python data/download_dataset.py
!python data/split_dataset.py

## 5. Obtain PlantDoc Dataset

In [ ]:
!python data/download_plantdoc.py
!python data/prepare_plantdoc.py

## 6. Verify Model 1 Base Checkpoint

In [ ]:
import pathlib, hashlib
m1_path = pathlib.Path('models/agrismart_best.pth')
assert m1_path.exists(), 'Model 1 checkpoint missing!'
print('Model 1 Size:', m1_path.stat().st_size / 1e6, 'MB')
print('Model 1 SHA256:', hashlib.sha256(m1_path.read_bytes()).hexdigest())

## 7. Prepare Model 2 Dataset (PlantVillage + PlantDoc Train Split)

In [ ]:
!python data/prepare_model2_dataset.py

## 8. Run Model 2 Pipeline Verification & 0-Leakage Guardrails

In [ ]:
!python data/verify_model2_pipeline.py

## 9. Run Training Dry-Run (Fast Verification)

In [ ]:
!python training/train_model2.py --dry-run

## 10. Execute Model 2 Full Training on GPU
Note: PlantDoc TEST remains 100% UNTOUCHED and is NOT evaluated during training.

In [ ]:
!python training/train_model2.py --epochs 25 --plantdoc-oversample-factor 5.0 --lr 1e-4

## 11. Verify Output Artifacts
Model 2 checkpoint saved to `models/agrismart_field_adapted_best.pth`.

In [ ]:
m2_path = pathlib.Path('models/agrismart_field_adapted_best.pth')
assert m2_path.exists(), 'Model 2 best checkpoint missing!'
print('Model 2 Size:', m2_path.stat().st_size / 1e6, 'MB')
!cat models/agrismart_field_adapted_metadata.json